In [1]:
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# 从您保存的 model.py 中导入模型和损失函数
from model import WavCapsNet, MarginLoss

In [2]:
# ==========================================
# 1. 超参数设置
# ==========================================
WINDOW_SIZE = 1024
STEP_SIZE = 512
BATCH_SIZE = 64
EPOCHS = 60
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ==========================================
# 2. 数据处理函数
# ==========================================
def sliding_window(data, window_size=1024, step_size=512):
    """滑动窗口切分一维数据"""
    data = data.flatten()
    num_samples = (len(data) - window_size) // step_size + 1
    samples = np.zeros((num_samples, window_size))
    for i in range(num_samples):
        start = i * step_size
        samples[i] = data[start : start + window_size]
    return samples

def load_and_preprocess_data():
    print("开始加载并预处理数据...")
    # 新增正常数据路径
    path_normal = '../../数据集/BJTU/正常/data_leftaxlebox_M0_G0_LA0_RA0_20Hz_-10kN.csv'
    path_inner = '../../数据集/BJTU/内圈/data_leftaxlebox_M0_G0_LA1_RA0_20Hz_-10kN.csv'
    path_outer = '../../数据集/BJTU/外圈/data_leftaxlebox_M0_G0_LA2_RA0_20Hz_-10kN.csv'
    path_roller = '../../数据集/BJTU/滚动体/data_leftaxlebox_M0_G0_LA3_RA0_20Hz_-10kN.csv'
    path_compound = '../../数据集/BJTU/外圈加滚动体/data_leftaxlebox_M0_G0_LA2+LA3_RA0_20Hz_-10kN.csv'

    # 读取数据
    data_normal = pd.read_csv(path_normal, usecols=['CH17']).values
    data_inner = pd.read_csv(path_inner, usecols=['CH17']).values
    data_outer = pd.read_csv(path_outer, usecols=['CH17']).values
    data_roller = pd.read_csv(path_roller, usecols=['CH17']).values
    data_compound = pd.read_csv(path_compound, usecols=['CH17']).values

    # 滑动窗口切分
    X_normal = sliding_window(data_normal, WINDOW_SIZE, STEP_SIZE)
    X_inner = sliding_window(data_inner, WINDOW_SIZE, STEP_SIZE)
    X_outer = sliding_window(data_outer, WINDOW_SIZE, STEP_SIZE)
    X_roller = sliding_window(data_roller, WINDOW_SIZE, STEP_SIZE)
    X_compound = sliding_window(data_compound, WINDOW_SIZE, STEP_SIZE)

    # 生成标签 (0: 正常, 1: 内圈, 2: 外圈, 3: 滚动体)
    y_normal = np.zeros(len(X_normal))
    y_inner = np.ones(len(X_inner)) * 1
    y_outer = np.ones(len(X_outer)) * 2
    y_roller = np.ones(len(X_roller)) * 3

    # 合并基础数据 (包含正常和单一故障)
    X_single = np.concatenate((X_normal, X_inner, X_outer, X_roller), axis=0)
    y_single = np.concatenate((y_normal, y_inner, y_outer, y_roller), axis=0)

    # 划分训练集和验证集 (80% 训练, 20% 验证)
    X_train, X_val, y_train, y_val = train_test_split(
        X_single, y_single, test_size=0.2, random_state=42, stratify=y_single
    )

# === 将 train.py 中的 StandardScaler 替换为以下代码 ===
    def instance_normalize(data):
        # 沿着时间轴 (axis=1) 独立计算每一个样本的均值和标准差
        mean = np.mean(data, axis=1, keepdims=True)
        std = np.std(data, axis=1, keepdims=True)
        # 加上 1e-8 防止除零错
        return (data - mean) / (std + 1e-8)

    X_train = instance_normalize(X_train)
    X_val = instance_normalize(X_val)
    X_compound = instance_normalize(X_compound)

    # 转换形状适配 PyTorch 1D 卷积: (Samples, Channels, Sequence_Length)
    X_train = X_train.reshape(-1, 1, WINDOW_SIZE)
    X_val = X_val.reshape(-1, 1, WINDOW_SIZE)
    X_compound = X_compound.reshape(-1, 1, WINDOW_SIZE)

    # 转换为 Tensor
    train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))
    compound_tensor = torch.FloatTensor(X_compound)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"训练集样本数: {len(X_train)}, 验证集样本数: {len(X_val)}, 复合故障测试集样本数: {len(X_compound)}")
    return train_loader, val_loader, compound_tensor



In [4]:
# ==========================================
# 3. 训练与验证过程
# ==========================================
def train_model():
    train_loader, val_loader, compound_tensor = load_and_preprocess_data()

    model = WavCapsNet().to(DEVICE)
    criterion = MarginLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
# === 新增：学习率调度器 (ReduceLROnPlateau) ===
    # 当验证集 Loss 连续 5 个 Epoch 不下降时，将学习率乘以 0.5
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    print(f"\n模型已加载到: {DEVICE}")
    print("开始训练...")

    for epoch in range(EPOCHS):
        # --- 训练阶段 ---
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

            optimizer.zero_grad()
            # 前向传播 (WavCapsNet 返回 probs 和 c_ij)
            probs, c_ij = model(batch_x)
            
            # 计算 Margin Loss
            loss = criterion(probs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_x.size(0)
            
            # 计算准确率 (模长最大的类别即为预测类别)
            preds = torch.argmax(probs, dim=1)
            correct_train += (preds == batch_y).sum().item()
            total_train += batch_x.size(0)

        epoch_train_loss = train_loss / total_train
        epoch_train_acc = correct_train / total_train

        # --- 验证阶段 ---
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                probs, c_ij = model(batch_x)
                loss = criterion(probs, batch_y)

                val_loss += loss.item() * batch_x.size(0)
                preds = torch.argmax(probs, dim=1)
                correct_val += (preds == batch_y).sum().item()
                total_val += batch_x.size(0)

        epoch_val_loss = val_loss / total_val
        epoch_val_acc = correct_val / total_val

        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")
        # === 新增：在每个 Epoch 结束时更新调度器 ===
        scheduler.step(epoch_val_loss)
    # ==========================================
    # 4. 复合故障解耦测试 (Zero-shot Testing)
    # ==========================================
    print("\n" + "="*50)
    print("开始进行复合故障 (外圈 + 滚动体) 解耦测试...")
    model.eval()
    
    # 为了防止显存溢出，我们分批次测试复合数据
    compound_loader = DataLoader(TensorDataset(compound_tensor), batch_size=BATCH_SIZE, shuffle=False)
    
    all_probs = []
    with torch.no_grad():
        for batch_x in compound_loader:
            batch_x = batch_x[0].to(DEVICE)
            probs, _ = model(batch_x)
            all_probs.append(probs.cpu().numpy())
            
    all_probs = np.concatenate(all_probs, axis=0)
    
    # 计算复合故障样本在四个胶囊上的平均激活概率 (模长)
    avg_probs = np.mean(all_probs, axis=0)
    
    print("\n复合故障测试集上的平均胶囊激活概率 (模长):")
    print(f" - [0] 正常状态胶囊激活度: {avg_probs[0]:.4f}  <-- 预期极低")
    print(f" - [1] 内圈故障胶囊激活度: {avg_probs[1]:.4f}  <-- 预期极低")
    print(f" - [2] 外圈故障胶囊激活度: {avg_probs[2]:.4f}  <-- 预期较高 (外圈+滚动体)")
    print(f" - [3] 滚动体故障胶囊激活度: {avg_probs[3]:.4f}  <-- 预期较高 (外圈+滚动体)")
    
    print("\n如果 [2] 和 [3] 的值显著高于 [0] 和 [1]，则证明 WavCapsNet 成功解耦了未见过的复合故障！")

if __name__ == '__main__':
    train_model()

开始加载并预处理数据...
训练集样本数: 3996, 验证集样本数: 1000, 复合故障测试集样本数: 1249

模型已加载到: cuda
开始训练...
Epoch [1/60] | Train Loss: 0.2258, Train Acc: 0.7520 | Val Loss: 0.1469, Val Acc: 0.8610
Epoch [2/60] | Train Loss: 0.1240, Train Acc: 0.8931 | Val Loss: 0.1204, Val Acc: 0.8900
Epoch [3/60] | Train Loss: 0.1024, Train Acc: 0.9252 | Val Loss: 0.1118, Val Acc: 0.8900
Epoch [4/60] | Train Loss: 0.0859, Train Acc: 0.9452 | Val Loss: 0.0969, Val Acc: 0.9230
Epoch [5/60] | Train Loss: 0.0709, Train Acc: 0.9592 | Val Loss: 0.0871, Val Acc: 0.9250
Epoch [6/60] | Train Loss: 0.0566, Train Acc: 0.9695 | Val Loss: 0.0751, Val Acc: 0.9380
Epoch [7/60] | Train Loss: 0.0455, Train Acc: 0.9810 | Val Loss: 0.0671, Val Acc: 0.9380
Epoch [8/60] | Train Loss: 0.0370, Train Acc: 0.9860 | Val Loss: 0.0600, Val Acc: 0.9500
Epoch [9/60] | Train Loss: 0.0297, Train Acc: 0.9905 | Val Loss: 0.0583, Val Acc: 0.9570
Epoch [10/60] | Train Loss: 0.0248, Train Acc: 0.9932 | Val Loss: 0.0526, Val Acc: 0.9620
Epoch [11/60] | Train Loss: 